In [1]:
import numpy as np
import pandas as pd

WORK_DIR = "/kaggle/working"

TRAIN_CSV = "/kaggle/input/competitions/document-image-classification-sd234504/train.csv"

train_df = pd.read_csv(TRAIN_CSV)
y = train_df["label"].values

N_CLASSES = 16
N_SPLITS = 5

# Load label asli
train_df = pd.read_csv(TRAIN_CSV)
y_meta = train_df["label"].values

# Load OOF gabungan
oof_dit = np.load("/kaggle/input/datasets/diftaradityapratama/dataset-level-1/oof_dit.npy")
oof_convnext = np.load("/kaggle/input/datasets/diftaradityapratama/dataset-level-1/oof_convnext.npy")

# Validasi dimensi
print("OOF DiT:", oof_dit.shape)
print("OOF ConvNeXt:", oof_convnext.shape)

assert oof_dit.shape == (len(y_meta), 16)
assert oof_convnext.shape == (len(y_meta), 16)

# Gabungkan probabilitas
X_meta = np.concatenate(
    [oof_dit, oof_convnext],
    axis=1
)

print("X_meta:", X_meta.shape)
print("y_meta:", y_meta.shape)

OOF DiT: (32003, 16)
OOF ConvNeXt: (32003, 16)
X_meta: (32003, 32)
y_meta: (32003,)


In [ ]:
from xgboost import XGBClassifier

best_xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=16,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

best_xgb_model.fit(X_meta, y_meta)

print("Meta-model berhasil dilatih ulang.")

In [ ]:
test_dit = np.load("/kaggle/input/datasets/diftaradityapratama/dataset-level-1/test_dit.npy")
test_convnext = np.load("/kaggle/input/datasets/diftaradityapratama/dataset-level-1/test_convnext.npy")

test_df = pd.read_csv("/kaggle/input/datasets/diftaradityapratama/urutan-test-df/output.csv")

sample_sub = pd.read_csv(
    "/kaggle/input/competitions/document-image-classification-sd234504/sample_submission.csv"
)

assert len(test_df) == len(test_dit) == len(test_convnext)

desired_order = sample_sub["id"].astype(str) + ".tif"

filename_to_index = pd.Series(
    np.arange(len(test_df)),
    index=test_df["filename"]
)

row_indices = filename_to_index.reindex(desired_order)

assert row_indices.notna().all(), "Ada filename yang tidak ditemukan."

row_indices = row_indices.to_numpy(dtype=int)

test_dit_aligned = test_dit[row_indices]
test_convnext_aligned = test_convnext[row_indices]

X_test_meta = np.concatenate(
    [test_dit_aligned, test_convnext_aligned],
    axis=1
)

print("X_test_meta:", X_test_meta.shape)

In [ ]:
test_pred = best_xgb_model.predict(X_test_meta)

submission = sample_sub.copy()
submission["label"] = test_pred.astype(int)

submission.to_csv(
    f"{WORK_DIR}/submission_reordered.csv",
    index=False
)

print(submission.head())
print("Shape:", submission.shape)
print("Distribusi prediksi:")
print(submission["label"].value_counts().sort_index())

# OPTUNA

In [6]:
import os
import json
import joblib
import optuna
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

print("Optuna:", optuna.__version__)
print("XGBoost:", xgb.__version__)

# Configuration
RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS = 30
N_CLASSES = 16

Optuna: 5.0.0
XGBoost: 3.4.1


In [7]:
# ==========================================
# CROSS VALIDATION
# ==========================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=2026
)


# ==========================================
# OPTUNA OBJECTIVE
# ==========================================

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 300, 1200, step=100
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 2, 9
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1, 12, log=True
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.65, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-2, 100.0, log=True
        ),

        "max_bin": trial.suggest_int(
            "max_bin", 128, 512, step=64
        )
    }

    model = XGBClassifier(
        **params,

        objective="multi:softprob",
        num_class=N_CLASSES,
        eval_metric="mlogloss",

        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    )

    scores = cross_val_score(
        model,
        X_meta,
        y_meta,
        cv=cv,
        scoring="accuracy",
        n_jobs=1
    )

    mean_accuracy = scores.mean()

    trial.set_user_attr(
        "fold_scores",
        scores.tolist()
    )

    return mean_accuracy

In [8]:
# ==========================================
# OPTUNA STUDY
# ==========================================

DB_PATH = f"{WORK_DIR}/optuna_xgb_accuracy.db"

study = optuna.create_study(
    study_name="xgb_meta_accuracy",
    direction="maximize",

    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE,
        n_startup_trials=10
    ),

    storage=f"sqlite:///{DB_PATH}",
    load_if_exists=True
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True
)

[I 2026-11-04 01:28:52,696] A new study created in RDB with name: xgb_meta_accuracy


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-11-04 01:31:41,924] Trial 0 finished with value: 0.8445144948054992 and parameters: {'n_estimators': 600, 'max_depth': 9, 'learning_rate': 0.08960785365368121, 'min_child_weight': 4.426505507739147, 'subsample': 0.7046065241548528, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 2.5378155082656657, 'max_bin': 384}. Best is trial 0 with value: 0.8445144948054992.
[I 2026-11-04 01:32:33,307] Trial 1 finished with value: 0.8447957740977973 and parameters: {'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.12106896936002161, 'min_child_weight': 1.6949324171244329, 'subsample': 0.7136387385224853, 'colsample_bytree': 0.6733618039413735, 'gamma': 1.5212112147976886, 'reg_alpha': 0.00052821153945323, 'reg_lambda': 0.5342937261279778, 'max_bin': 256}. Best is trial 1 with value: 0.8447957740977973.
[I 2026-11-04 01:34:46,647] Trial 2 finished with value: 0.8468268971645057 and parameters: {'n_estimators': 900, 'ma

In [9]:
print("=" * 50)
print("OPTUNA RESULTS")
print("=" * 50)

print("Best Trial:", study.best_trial.number)
print("Best CV Accuracy:", study.best_value)

print("\nBest Parameters:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

OPTUNA RESULTS
Best Trial: 28
Best CV Accuracy: 0.84942027612873

Best Parameters:
n_estimators: 400
max_depth: 5
learning_rate: 0.01147065327432187
min_child_weight: 4.670280143819791
subsample: 0.7453733398030049
colsample_bytree: 0.7099211473454187
gamma: 1.0974497257821267
reg_alpha: 6.01950290107361e-07
reg_lambda: 1.3911992427663566
max_bin: 256


In [10]:
best_params = study.best_params.copy()

best_xgb_model = XGBClassifier(
    **best_params,

    objective="multi:softprob",
    num_class=N_CLASSES,
    eval_metric="mlogloss",

    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)

best_xgb_model.fit(X_meta, y_meta)

print("Final XGBoost berhasil dilatih.")

Final XGBoost berhasil dilatih.


In [11]:
joblib.dump(
    best_xgb_model,
    f"{WORK_DIR}/xgb_meta_optuna.joblib"
)

['/kaggle/working/xgb_meta_optuna.joblib']

In [12]:
# Load test probabilities
test_dit = np.load("/kaggle/input/datasets/diftaradityapratama/dataset-level-1/test_dit.npy")
test_convnext = np.load("/kaggle/input/datasets/diftaradityapratama/dataset-level-1/test_convnext.npy")

# Load urutan test_df yang digunakan saat inferensi
test_df = pd.read_csv("/kaggle/input/datasets/diftaradityapratama/urutan-test-df/output.csv")

# Load sample submission
sample_sub = pd.read_csv("/kaggle/input/competitions/document-image-classification-sd234504/sample_submission.csv")

assert len(test_df) == len(test_dit) == len(test_convnext)

# Periksa filename
assert test_df["filename"].is_unique

# Buat filename berdasarkan urutan sample submission
desired_files = (
    sample_sub["id"].astype(str) + ".tif"
)

# Mapping posisi filename ke array
filename_to_index = pd.Series(
    np.arange(len(test_df)),
    index=test_df["filename"].astype(str)
)

row_indices = filename_to_index.reindex(
    desired_files
)

assert row_indices.notna().all(), (
    "Ada filename yang tidak ditemukan!"
)

row_indices = row_indices.to_numpy(dtype=int)

# Reorder test predictions
test_dit_aligned = test_dit[row_indices]
test_convnext_aligned = test_convnext[row_indices]

# Gabungkan 32 fitur test
X_test_meta = np.concatenate(
    [
        test_dit_aligned,
        test_convnext_aligned
    ],
    axis=1
)

print("X_test_meta:", X_test_meta.shape)

assert X_test_meta.shape == (
    len(sample_sub), 32
)

X_test_meta: (7994, 32)


In [13]:
# Prediksi kelas test
test_pred = best_xgb_model.predict(X_test_meta)

# Buat submission
submission = sample_sub.copy()

submission["label"] = test_pred.astype(int)

# Validasi
assert len(submission) == 7994
assert submission["label"].between(0, 15).all()
assert submission.isna().sum().sum() == 0

# Simpan
submission.to_csv(
    f"{WORK_DIR}/submission_optuna.csv",
    index=False
)

print(submission.head())
print("\nDistribusi prediksi:")
print(submission["label"].value_counts().sort_index())

print("\nSubmission berhasil dibuat!")

           id  label
0  test_00001      0
1  test_00002      0
2  test_00003      1
3  test_00004      4
4  test_00005     13

Distribusi prediksi:
label
0     507
1     487
2     501
3     544
4     488
5     488
6     493
7     473
8     468
9     466
10    531
11    476
12    497
13    505
14    572
15    498
Name: count, dtype: int64

Submission berhasil dibuat!
